# OpenPlaque experimental outer-wall / PAV prototype

Research prototype only. The candidate outer wall is **not clinically validated**. Review the overlays before interpreting any PAV value.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!rm -rf /content/OpenPlaque
!git clone -b pav-outer-wall-prototype https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
!pip -q install -e /content/OpenPlaque
print('OpenPlaque PAV prototype installed.')

## Locate CCTA and artery masks
The notebook checks the OpenPlaque Drive folders used by the existing analysis. If it cannot identify the CCTA automatically, set `CCTA_PATH` manually in the next cell.

In [ ]:
from pathlib import Path

DRIVE = Path('/content/drive/MyDrive')
MASK_DIR_CANDIDATES = [
    DRIVE / 'OpenPlaque/UCLA_Plaque_Type_Estimates/nnunet_masks',
    DRIVE / 'OpenPlaque/UCLA_Plaque_Context_Verification/nnunet_masks',
]

MASK_DIR = next((p for p in MASK_DIR_CANDIDATES if all((p / f'{a}.nii.gz').exists() for a in ['LAD','LCX','RCA'])), None)
if MASK_DIR is None:
    raise FileNotFoundError('Could not find LAD.nii.gz, LCX.nii.gz, and RCA.nii.gz in the expected OpenPlaque folders.')

print('Mask directory:', MASK_DIR)
for a in ['LAD','LCX','RCA']:
    print(a, '->', MASK_DIR / f'{a}.nii.gz')

# Set this manually if automatic discovery below does not pick the correct original CCTA.
CCTA_PATH = None

search_roots = [
    DRIVE / 'OpenPlaque/UCLA_Plaque_Type_Estimates',
    DRIVE / 'OpenPlaque/UCLA_Plaque_Context_Verification',
    DRIVE / 'OpenPlaque',
]

if CCTA_PATH is None:
    candidates = []
    for root in search_roots:
        if root.exists():
            for pattern in ('*.nii.gz', '*.nii'):
                candidates.extend(root.rglob(pattern))
    candidates = [p for p in candidates if p.name not in {'LAD.nii.gz','LCX.nii.gz','RCA.nii.gz'} and 'mask' not in p.name.lower() and 'label' not in p.name.lower()]
    print('Possible CCTA volumes:')
    for i, p in enumerate(candidates[:30]):
        print(f'[{i}] {p}')
    if len(candidates) == 1:
        CCTA_PATH = candidates[0]

print('CCTA_PATH =', CCTA_PATH)
print('If this is None or incorrect, set CCTA_PATH = Path("/content/drive/MyDrive/.../your_cta.nii.gz") and rerun this cell.')

## Load images and verify geometry

In [ ]:
import numpy as np
import SimpleITK as sitk
from openplaque.pav import estimate_pav_from_labels, show_pav_overlay

if CCTA_PATH is None:
    raise ValueError('Set CCTA_PATH to the original CCTA NIfTI file before continuing.')
CCTA_PATH = Path(CCTA_PATH)

ct_img = sitk.ReadImage(str(CCTA_PATH))
ct = sitk.GetArrayFromImage(ct_img)
spacing = ct_img.GetSpacing()

print('CCTA:', CCTA_PATH)
print('CT array shape (z,y,x):', ct.shape)
print('Spacing (x,y,z) mm:', spacing)

masks = {}
for artery in ['LAD','LCX','RCA']:
    img = sitk.ReadImage(str(MASK_DIR / f'{artery}.nii.gz'))
    arr = sitk.GetArrayFromImage(img)
    masks[artery] = arr
    print(artery, 'shape=', arr.shape, 'spacing=', img.GetSpacing(), 'labels=', np.unique(arr))
    if arr.shape != ct.shape:
        raise ValueError(f'{artery} mask shape {arr.shape} does not match CT shape {ct.shape}. These files are not directly aligned.')


## Run experimental PAV estimate
Start with a conservative 2.0 mm maximum expansion and -30 HU fat threshold. These parameters are intentionally exposed for review, not treated as validated defaults.

In [ ]:
MAX_WALL_THICKNESS_MM = 2.0
FAT_THRESHOLD_HU = -30.0

results = {}
for artery, mask in masks.items():
    result = estimate_pav_from_labels(
        volume=ct,
        mask=mask,
        spacing=spacing,
        max_wall_thickness_mm=MAX_WALL_THICKNESS_MM,
        fat_threshold_hu=FAT_THRESHOLD_HU,
    )
    results[artery] = result
    print('\n' + artery)
    result.summary()

## Visual review: highest-plaque slice for each artery

In [ ]:
import matplotlib.pyplot as plt

for artery in ['LAD','LCX','RCA']:
    print(artery)
    fig, ax = show_pav_overlay(ct, masks[artery], results[artery].outer_wall_mask)
    plt.show()

## Visual review: three plaque-containing slices per artery

In [ ]:
for artery in ['LAD','LCX','RCA']:
    mask = masks[artery]
    counts = np.sum(mask == 2, axis=(1,2))
    zs = np.where(counts > 0)[0]
    if len(zs) == 0:
        print(artery, ': no plaque-labeled slices')
        continue
    picks = np.unique(np.linspace(0, len(zs)-1, min(3, len(zs)), dtype=int))
    for idx in picks:
        z = int(zs[idx])
        print(f'{artery} z={z}, plaque voxels={counts[z]}')
        fig, ax = show_pav_overlay(ct, mask, results[artery].outer_wall_mask, z=z)
        plt.show()

## Summary table and whole-heart experimental PAV

In [ ]:
import pandas as pd

rows = []
for artery, r in results.items():
    rows.append({
        'artery': artery,
        'plaque_volume_mm3': r.plaque_volume_mm3,
        'outer_vessel_volume_mm3': r.outer_vessel_volume_mm3,
        'experimental_pav_percent': r.pav_percent,
    })
df = pd.DataFrame(rows)
display(df.round(3))

total_plaque = df.plaque_volume_mm3.sum()
total_outer = df.outer_vessel_volume_mm3.sum()
whole_pav = 100 * total_plaque / total_outer if total_outer else 0
print(f'Total plaque volume: {total_plaque:.2f} mm^3')
print(f'Total candidate outer-vessel volume: {total_outer:.2f} mm^3')
print(f'Whole-heart experimental PAV: {whole_pav:.2f}%')
print('Do not interpret this clinically until outer-wall contours are validated.')

## Save candidate masks and CSV to Drive

In [ ]:
OUT_DIR = DRIVE / 'OpenPlaque/PAV_Outer_Wall_Prototype'
OUT_DIR.mkdir(parents=True, exist_ok=True)

for artery, r in results.items():
    out_img = sitk.GetImageFromArray(r.outer_wall_mask.astype(np.uint8))
    out_img.CopyInformation(ct_img)
    sitk.WriteImage(out_img, str(OUT_DIR / f'{artery}_candidate_outer_wall.nii.gz'))

df.to_csv(OUT_DIR / 'experimental_pav_by_artery.csv', index=False)
print('Saved results to:', OUT_DIR)